In [ ]:
import json
import re
from pathlib import Path

# ============================================================
# CONFIGURATION
# ============================================================

INPUT_FILE = Path("30m_polygons_simplified_10.js")
OUTPUT_DIR = Path("30m_polygons")

VARIABLE_NAME = "json_30m_polygons_simplified_10"


# ============================================================
# FUNCTIONS
# ============================================================

def safe_filename(name):
    """
    Convert district name into a safe filename.
    Example:
        'Nkhata Bay' -> 'Nkhata_Bay'
    """
    name = str(name).strip()

    # Replace anything other than letters/numbers with _
    name = re.sub(r"[^A-Za-z0-9_-]+", "_", name)

    return name.strip("_")


def extract_json_from_js(filepath):
    """
    Extract the JSON object from:

        var json_30m_polygons_simplified_10 = {...};

    and convert it into a Python dictionary.
    """

    print(f"Reading: {filepath}")

    text = filepath.read_text(encoding="utf-8")

    # Find the first { after the variable declaration
    start = text.find("{")

    if start == -1:
        raise ValueError("Could not find the beginning of the JSON object.")

    # Find the final }
    end = text.rfind("}")

    if end == -1:
        raise ValueError("Could not find the end of the JSON object.")

    json_text = text[start:end + 1]

    try:
        data = json.loads(json_text)
    except json.JSONDecodeError as e:
        raise ValueError(
            f"Could not parse JavaScript data as JSON.\n"
            f"Error: {e}"
        )

    return data


def write_district_file(district, features, output_dir):
    """
    Write one JavaScript FeatureCollection for a district.
    """

    safe_name = safe_filename(district)

    output_file = output_dir / f"{safe_name}.js"

    district_data = {
        "type": "FeatureCollection",
        "name": f"30m_polygons_{safe_name}",
        "crs": {
            "type": "name",
            "properties": {
                "name": "urn:ogc:def:crs:OGC:1.3:CRS84"
            }
        },
        "features": features
    }

    json_string = json.dumps(
        district_data,
        ensure_ascii=False,
        separators=(",", ":")
    )

    js_content = (
        f"var json_100m_{safe_name} = "
        f"{json_string};\n"
    )

    output_file.write_text(
        js_content,
        encoding="utf-8"
    )

    return output_file


# ============================================================
# MAIN
# ============================================================

def main():

    # Create output directory
    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    # --------------------------------------------------------
    # Read source file
    # --------------------------------------------------------

    data = extract_json_from_js(INPUT_FILE)

    features = data.get("features", [])

    print()
    print(f"Total features: {len(features):,}")
    print()

    # --------------------------------------------------------
    # Group features by district
    # --------------------------------------------------------

    districts = {}

    missing_district = []

    for feature in features:

        properties = feature.get("properties", {})

        district = properties.get("adm2_name")

        if district is None or str(district).strip() == "":
            missing_district.append(feature)
            continue

        district = str(district).strip()

        districts.setdefault(
            district,
            []
        ).append(feature)

    # --------------------------------------------------------
    # Write district files
    # --------------------------------------------------------

    print("Creating district files...")
    print()

    total_written = 0

    for district, district_features in sorted(districts.items()):

        output_file = write_district_file(
            district,
            district_features,
            OUTPUT_DIR
        )

        total_written += len(district_features)

        print(
            f"{district:<20} "
            f"{len(district_features):>10,} features "
            f"-> {output_file.name}"
        )

    # --------------------------------------------------------
    # Handle missing districts
    # --------------------------------------------------------

    if missing_district:

        output_file = write_district_file(
            "Unknown_District",
            missing_district,
            OUTPUT_DIR
        )

        print()
        print(
            f"WARNING: {len(missing_district):,} "
            f"features have no adm2_name."
        )

        print(
            f"Written to: {output_file}"
        )

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    print()
    print("=" * 60)
    print("COMPLETE")
    print("=" * 60)

    print(
        f"Districts created: {len(districts)}"
    )

    print(
        f"Features written: {total_written:,}"
    )

    print(
        f"Output directory: {OUTPUT_DIR.resolve()}"
    )


if __name__ == "__main__":
    main()